# PyTorch — A Complete Beginner's Tutorial
### Tensors · Autograd · Neural Networks · Training Loops · GPU

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)  
**Repo:** `computational-science-tutorials`

---

## What is PyTorch?

PyTorch is an open-source **deep learning framework** developed by Meta AI. It is the most popular framework for research and is increasingly dominant in production.

**Why PyTorch over other frameworks?**
- **Pythonic** — feels like NumPy with GPU support and automatic differentiation
- **Dynamic computation graphs** — graphs are built at runtime (easy to debug)
- **Autograd** — automatic gradient computation for any tensor operation
- **Ecosystem** — PyTorch Geometric, HuggingFace, Lightning, all built on PyTorch

## What you will learn

| Section | Topics |
|---------|--------|
| 1. Tensors | Creation, shapes, dtypes, GPU |
| 2. Tensor ops | Indexing, slicing, math, broadcasting |
| 3. Autograd | Gradients, computation graphs, backprop |
| 4. `nn.Module` | Layers, parameters, custom modules |
| 5. Activations | ReLU, GELU, Sigmoid, Softmax |
| 6. Loss functions | BCE, CrossEntropy, MSE, Focal |
| 7. Optimisers | SGD, Adam, AdamW, schedulers |
| 8. Training loop | Full pattern with early stopping |
| 9. DataLoader | Dataset class, batching, shuffling |
| 10. Saving/loading | Checkpoints, model export |
| 11. GPU workflow | `.to(device)`, memory management |
| 12. Debugging | Common errors and fixes |

---
## Section 1 — Tensors: The Building Block

In [ ]:
# ── Install & import ──────────────────────────────────────────────────────────
# !pip install torch torchvision

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device:    {device}")

In [ ]:
# ── 1.1 Creating tensors ─────────────────────────────────────────────────────
# A tensor = multi-dimensional array (like numpy ndarray) + GPU support + autograd

# From Python list
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("From list:       ", a)
print("  shape:", a.shape, "  dtype:", a.dtype)

# 2D tensor (matrix)
b = torch.tensor([[1, 2, 3],
                   [4, 5, 6]], dtype=torch.float32)
print("\n2D tensor:\n", b)
print("  shape:", b.shape)  # [2, 3] = 2 rows × 3 cols

# Built-in constructors
zeros   = torch.zeros(3, 4)           # all zeros
ones    = torch.ones(2, 3)            # all ones
rand    = torch.rand(3, 3)            # uniform [0, 1)
randn   = torch.randn(3, 3)           # standard normal
eye     = torch.eye(4)                # identity matrix
arange  = torch.arange(0, 10, 2)      # [0, 2, 4, 6, 8]
linsp   = torch.linspace(0, 1, 5)     # [0.00, 0.25, 0.50, 0.75, 1.00]

print(f"\nzeros {zeros.shape}:\n{zeros}")
print(f"\narange: {arange}")
print(f"linspace: {linsp}")

In [ ]:
# ── 1.2 Tensor shapes and dtypes ─────────────────────────────────────────────
t = torch.randn(2, 3, 4)   # 3D tensor: batch=2, rows=3, cols=4

print("Shape:         ", t.shape)          # torch.Size([2, 3, 4])
print("Num dimensions:", t.ndim)           # 3
print("Total elements:", t.numel())        # 2×3×4 = 24
print("dtype:         ", t.dtype)          # torch.float32
print("device:        ", t.device)         # cpu

# Common dtypes
print("\nDtype examples:")
print("  float32 (default):", torch.tensor([1.0]).dtype)
print("  float16 (half):   ", torch.tensor([1.0], dtype=torch.float16).dtype)
print("  int64   (long):   ", torch.tensor([1]).dtype)
print("  bool:             ", torch.tensor([True]).dtype)

# Type casting
x_int   = torch.tensor([1, 2, 3])          # int64 by default
x_float = x_int.float()                     # → float32
x_half  = x_float.half()                    # → float16 (for GPU efficiency)
print(f"\nCasting: {x_int.dtype} → {x_float.dtype} → {x_half.dtype}")

---
## Section 2 — Tensor Operations

In [ ]:
# ── 2.1 Indexing and slicing — identical to NumPy ───────────────────────────
x = torch.tensor([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]], dtype=torch.float32)

print("Full tensor:\n", x)
print("\nRow 0:          ", x[0])             # first row
print("Col 1:          ", x[:, 1])            # second column
print("Element [1,2]:  ", x[1, 2])            # scalar
print("Submatrix [0:2, 1:3]:\n", x[0:2, 1:3])  # 2×2 slice
print("Boolean mask:   ", x[x > 5])           # [6, 7, 8, 9]

In [ ]:
# ── 2.2 Shape manipulation ───────────────────────────────────────────────────
x = torch.arange(12, dtype=torch.float32)
print("Original:     ", x.shape)         # [12]

# reshape / view: change shape without copying data
a = x.reshape(3, 4)       # [3, 4]
b = x.view(2, 6)           # [2, 6] — like reshape but requires contiguous memory
c = x.reshape(2, 2, 3)    # [2, 2, 3]
print("reshape(3,4): ", a.shape)
print("view(2,6):    ", b.shape)

# squeeze / unsqueeze: add or remove size-1 dimensions
d = a.unsqueeze(0)   # add dim at position 0: [1, 3, 4]
e = d.squeeze(0)     # remove dim 0:          [3, 4]
print("unsqueeze(0): ", d.shape, "→ squeeze(0):", e.shape)

# Common in ML: add batch dimension
sample = torch.randn(128)          # one sample
batch  = sample.unsqueeze(0)       # [1, 128] — fake batch of 1
print("\nSingle sample → batch:", sample.shape, "→", batch.shape)

# flatten: collapse all dims to 1D
f = torch.randn(4, 3, 2)
print("flatten:", f.flatten().shape)       # [24]

# permute: rearrange axes
img = torch.randn(3, 64, 64)    # [channels, H, W]
img_hwc = img.permute(1, 2, 0)  # [H, W, channels] — for matplotlib
print("permute:", img.shape, "→", img_hwc.shape)

In [ ]:
# ── 2.3 Mathematical operations and broadcasting ─────────────────────────────
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Element-wise operations
print("a + b  :", a + b)          # [5, 7, 9]
print("a * b  :", a * b)          # [4, 10, 18]
print("a ** 2 :", a ** 2)         # [1, 4, 9]
print("a / b  :", (a / b).round(decimals=3))

# Matrix multiplication — the most important operation in deep learning!
W = torch.randn(3, 4)   # weight matrix [in=3, out=4]
x = torch.randn(5, 3)   # batch of 5 samples, each with 3 features

y = x @ W               # [5, 3] × [3, 4] = [5, 4]  (matmul)
# Equivalent: torch.matmul(x, W) or torch.mm(x, W) for 2D
print(f"\nMatrix multiply: {x.shape} @ {W.shape} = {y.shape}")

# Broadcasting: automatic shape expansion
# [3, 4] + [4] → [3, 4]  (bias added to each row)
bias   = torch.tensor([0.1, 0.2, 0.3, 0.4])
result = y + bias   # broadcasts [5, 4] + [4] → [5, 4]
print(f"Broadcasting:   {y.shape} + {bias.shape} → {result.shape}")

# Reduction operations
t = torch.randn(3, 4)
print(f"\nmean:           {t.mean():.4f}  (all elements)")
print(f"mean(dim=0):    {t.mean(dim=0)}  (mean over rows)")
print(f"mean(dim=1):    {t.mean(dim=1)}  (mean over cols)")
print(f"sum, max, min:  {t.sum():.2f}, {t.max():.2f}, {t.min():.2f}")
print(f"argmax:         {t.argmax()}  (flat index of max)")

In [ ]:
# ── 2.4 NumPy ↔ PyTorch conversion ──────────────────────────────────────────
# Very common when interfacing with sklearn, pandas, matplotlib

# NumPy → PyTorch
np_arr    = np.array([1.0, 2.0, 3.0])
pt_tensor = torch.from_numpy(np_arr)      # shares memory — no copy!
pt_tensor2= torch.tensor(np_arr)          # copies memory
print("NumPy → PyTorch:", pt_tensor, pt_tensor.dtype)

# PyTorch → NumPy
pt   = torch.tensor([4.0, 5.0, 6.0])
arr  = pt.numpy()                          # .numpy() — works on CPU only
print("PyTorch → NumPy:", arr, type(arr))

# For GPU tensors: must detach and move to CPU first
# arr = pt_gpu.detach().cpu().numpy()

# Get scalar value from 0-dimensional tensor
loss = torch.tensor(0.3456)
print(f"\nScalar tensor: {loss}  → Python float: {loss.item():.4f}")

---
## Section 3 — Autograd: Automatic Differentiation

Autograd is PyTorch's engine for computing gradients automatically. Every tensor operation is recorded in a **computation graph**, and `loss.backward()` traverses it in reverse (backpropagation).

In [ ]:
# ── 3.1 Basic gradient computation ──────────────────────────────────────────
# requires_grad=True → PyTorch tracks all operations on this tensor

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1   # y = x² + 2x + 1

# dy/dx = 2x + 2 = 2(3) + 2 = 8  at x=3
y.backward()              # compute gradients
print(f"x = {x.item()}, y = {y.item()}")
print(f"dy/dx = {x.grad.item()}  (expected: 2*3 + 2 = 8 ✓)")

In [ ]:
# ── 3.2 Gradients through a simple network ───────────────────────────────────
# Simulate one step of gradient descent manually

# Parameters (learnable)
W = torch.randn(2, 3, requires_grad=True)   # weight matrix
b = torch.randn(3,    requires_grad=True)   # bias

# Input (not learnable)
x = torch.tensor([[1.0, 2.0]])

# Forward pass
y_pred = x @ W + b          # linear layer: [1,2] @ [2,3] + [3] = [1,3]
y_true = torch.tensor([[0.5, -0.5, 1.0]])
loss   = ((y_pred - y_true) ** 2).mean()   # MSE loss

print(f"y_pred: {y_pred.data}")
print(f"loss:   {loss.item():.4f}")

# Backward pass — compute dL/dW and dL/db
loss.backward()

print(f"\ndL/dW:\n{W.grad}")
print(f"dL/db: {b.grad}")

# Manual gradient descent step (lr = 0.01)
lr = 0.01
with torch.no_grad():       # don't track this update in the graph!
    W -= lr * W.grad
    b -= lr * b.grad

# Always zero gradients before next backward pass!
W.grad.zero_()
b.grad.zero_()
print("\nManual gradient descent step done.")
print("In practice, use optimiser.zero_grad() instead of .grad.zero_()")

In [ ]:
# ── 3.3 Stopping gradient flow ───────────────────────────────────────────────
# You often need to stop gradients for evaluation, frozen layers, etc.

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Method 1: torch.no_grad() context manager (most common)
with torch.no_grad():
    y = x * 2   # y does not participate in autograd
print(f"no_grad: y.requires_grad = {y.requires_grad}")

# Method 2: .detach() — detach tensor from computation graph
z = (x * 3).detach()
print(f"detach:  z.requires_grad = {z.requires_grad}")

# Method 3: requires_grad_(False) — turn off for a specific tensor
W = torch.randn(3, 2, requires_grad=True)
W.requires_grad_(False)   # freeze this layer
print(f"frozen:  W.requires_grad = {W.requires_grad}")

print("\nRule: use torch.no_grad() for all evaluation / inference code")

---
## Section 4 — Building Neural Networks with `nn.Module`

All neural networks in PyTorch inherit from `nn.Module`. This provides:
- Automatic parameter tracking (`.parameters()`)
- State dict (saving/loading)
- Module nesting (.children(), .named_modules())

In [ ]:
# ── 4.1 A simple MLP from scratch ────────────────────────────────────────────
class SimpleMLP(nn.Module):
    """
    Multi-Layer Perceptron.
    in_features → hidden → hidden → out_features

    The __init__ method defines the LAYERS (learnable parameters).
    The forward method defines the COMPUTATION (how data flows through).
    """
    def __init__(self, in_features=128, hidden=64, out_features=1, dropout=0.3):
        super().__init__()   # always call super().__init__()

        # nn.Linear(in, out) = y = xW^T + b
        self.fc1     = nn.Linear(in_features, hidden)
        self.fc2     = nn.Linear(hidden, hidden)
        self.fc3     = nn.Linear(hidden, out_features)

        # Batch normalisation: normalises each mini-batch
        self.bn1     = nn.BatchNorm1d(hidden)
        self.bn2     = nn.BatchNorm1d(hidden)

        # Dropout: randomly zero out neurons during training
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        """
        x: [batch_size, in_features]
        returns: [batch_size, out_features]
        """
        # Layer 1: linear → batchnorm → relu → dropout
        x = self.dropout(F.relu(self.bn1(self.fc1(x))))
        # Layer 2
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))
        # Output layer (no activation — use loss function's built-in)
        return self.fc3(x)


model = SimpleMLP(in_features=10, hidden=32, out_features=1)
print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Test with a batch
x = torch.randn(16, 10)    # batch of 16, each with 10 features
y = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")

In [ ]:
# ── 4.2 Sequential API — for simple linear stacks ────────────────────────────
# nn.Sequential is a shorthand when layers are applied in order

model_seq = nn.Sequential(
    nn.Linear(10, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Linear(32, 1),
)

x = torch.randn(8, 10)
print("Sequential output:", model_seq(x).shape)

# ModuleList: use when you need dynamic or looped layers
class DynamicMLP(nn.Module):
    def __init__(self, sizes):  # e.g. sizes = [128, 64, 32, 16, 1]
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]
        )
    def forward(self, x):
        for layer in self.layers[:-1]:
            x = F.relu(layer(x))
        return self.layers[-1](x)   # no activation on last layer

dyn = DynamicMLP([10, 64, 32, 1])
print("Dynamic MLP output:", dyn(torch.randn(5, 10)).shape)

---
## Section 5 — Activation Functions

In [ ]:
# ── 5.1 Common activations with plots ────────────────────────────────────────
x = torch.linspace(-4, 4, 200)

activations = {
    'ReLU':    F.relu(x),               # max(0, x) — default choice
    'GELU':    F.gelu(x),               # Gaussian error linear unit — Transformers
    'SiLU':    F.silu(x),               # Swish = x·σ(x) — EfficientNet, modern CNNs
    'Sigmoid': torch.sigmoid(x),        # (0,1) — binary classification output
    'Tanh':    torch.tanh(x),           # (-1,1) — RNNs, normalisation layers
    'LeakyReLU': F.leaky_relu(x, 0.1), # fixes dying ReLU problem
    'ELU':     F.elu(x),               # smooth alternative to ReLU
    'Softplus':F.softplus(x),          # smooth ReLU
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
colours = ['#1565C0','#E74C3C','#27AE60','#E67E22','#8E44AD','#1ABC9C','#D35400','#C0392B']

for ax, (name, y), c in zip(axes.flat, activations.items(), colours):
    ax.plot(x.numpy(), y.detach().numpy(), color=c, lw=2.5)
    ax.axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
    ax.axvline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
    ax.set_title(name, fontsize=12, fontweight='bold', color=c)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.grid(True, alpha=0.3); ax.set_ylim(-2, 4)

plt.suptitle('Activation Functions in PyTorch', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("\nWhen to use which:")
print("  ReLU        → hidden layers (default, simple)")
print("  GELU        → Transformer hidden layers")
print("  SiLU/Swish  → EfficientNet, modern CNNs")
print("  Sigmoid     → binary output (but use BCEWithLogitsLoss, not manual sigmoid!)")
print("  Softmax     → multiclass output (but use CrossEntropyLoss, not manual softmax!)")
print("  Tanh        → RNN/LSTM gates, normalised features")

---
## Section 6 — Loss Functions

In [ ]:
# ── 6.1 Loss functions for each task type ────────────────────────────────────
print("Binary classification — BCEWithLogitsLoss")
print("─" * 50)
# BCEWithLogitsLoss = Sigmoid + BinaryCrossEntropy in one numerically stable operation
# NEVER do: loss = BCELoss(sigmoid(logits), targets) — use BCEWithLogitsLoss directly!
criterion = nn.BCEWithLogitsLoss()
logits  = torch.tensor([2.0, -1.0, 0.5, -2.0])   # model output (raw)
targets = torch.tensor([1.0,  0.0, 1.0,  0.0])   # ground truth
loss = criterion(logits, targets)
print(f"  logits:  {logits.tolist()}")
print(f"  targets: {targets.tolist()}")
print(f"  loss:    {loss.item():.4f}")

# Class-weighted BCE for imbalanced datasets (crucial for toxicity!)
# Weight > 1 penalises missing positives more
pos_weight = torch.tensor([3.0])   # 3× penalty for missing a positive
criterion_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
loss_w = criterion_weighted(logits, targets)
print(f"  weighted loss (pos_weight=3): {loss_w.item():.4f}")

print("\nMulti-class classification — CrossEntropyLoss")
print("─" * 50)
# CrossEntropyLoss = LogSoftmax + NLLLoss
# targets are INTEGER class indices (not one-hot!)
criterion_ce = nn.CrossEntropyLoss()
logits_mc = torch.tensor([[2.0, 0.5, -1.0],
                           [0.3, 3.1,  0.2]])   # [batch=2, classes=3]
targets_mc = torch.tensor([0, 1])               # class 0 and class 1
loss_mc = criterion_ce(logits_mc, targets_mc)
print(f"  loss: {loss_mc.item():.4f}")

print("\nRegression — MSELoss and MAELoss")
print("─" * 50)
y_pred = torch.tensor([2.5, 0.0, 2.1, 7.8])
y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])
print(f"  MSE (L2): {nn.MSELoss()(y_pred, y_true).item():.4f}")
print(f"  MAE (L1): {nn.L1Loss()(y_pred, y_true).item():.4f}")
print(f"  Huber:    {nn.HuberLoss()(y_pred, y_true).item():.4f}  (robust to outliers)")

---
## Section 7 — Optimisers & Learning Rate Schedulers

In [ ]:
# ── 7.1 Optimisers ───────────────────────────────────────────────────────────
from torch.optim import SGD, Adam, AdamW, RMSprop
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau, OneCycleLR

model = SimpleMLP(10, 32, 1)

# SGD — classic, requires careful lr tuning
opt_sgd  = SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# Adam — adaptive learning rate per parameter, most common default
opt_adam = Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8)

# AdamW — Adam + decoupled weight decay (correct L2 regularisation)
# → use this by default for modern networks
opt_adamw = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

print("Optimisers created:")
print(f"  SGD   : lr={opt_sgd.param_groups[0]['lr']}")
print(f"  Adam  : lr={opt_adam.param_groups[0]['lr']}")
print(f"  AdamW : lr={opt_adamw.param_groups[0]['lr']}")

# ── Learning rate schedulers ──────────────────────────────────────────────────
# Schedulers reduce lr over training → better convergence

optimiser = AdamW(model.parameters(), lr=3e-4)

# 1. CosineAnnealingLR: smoothly decreases lr following a cosine curve
sched_cos  = CosineAnnealingLR(optimiser, T_max=100, eta_min=1e-6)

# 2. ReduceLROnPlateau: reduces lr when val loss stops improving
sched_plat = ReduceLROnPlateau(optimiser, patience=5, factor=0.5)

# 3. OneCycleLR: warmup → peak → cosine decay (superconvergence)
sched_one  = OneCycleLR(optimiser, max_lr=1e-3, total_steps=100)

# Visualise OneCycleLR schedule
opt_vis = Adam(model.parameters(), lr=1e-3)
sched   = OneCycleLR(opt_vis, max_lr=1e-2, total_steps=100, pct_start=0.3)
lrs = []
for _ in range(100):
    lrs.append(opt_vis.param_groups[0]['lr'])
    sched.step()

plt.figure(figsize=(9, 4))
plt.plot(lrs, color='#E74C3C', lw=2.5)
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.title('OneCycleLR Schedule — Warmup → Peak → Cosine Decay', fontweight='bold')
plt.grid(True, alpha=0.35); plt.tight_layout(); plt.show()
print("OneCycleLR: 30% warmup, 70% cosine decay")

---
## Section 8 — The Complete Training Loop

This is the pattern you will use 90% of the time. Learn it by heart.

In [ ]:
# ── 8.1 Dataset and DataLoader setup ─────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader, random_split

class MoleculeDataset(Dataset):
    """
    Custom Dataset: wraps feature matrix X and labels y.
    Must implement __len__ and __getitem__.
    """
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)           # number of samples

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # one (features, label) pair


# Generate synthetic QSAR data (replace with your descriptor matrix + labels)
np.random.seed(42)
N      = 800
n_feat = 50
X_all  = np.random.randn(N, n_feat).astype(np.float32)
# Binary label: correlated with first 3 features
score  = X_all[:, 0] - X_all[:, 1] * 0.5 + X_all[:, 2] * 0.8
y_all  = (score > np.percentile(score, 55)).astype(np.float32)

# Split: 70% train, 15% val, 15% test
dataset = MoleculeDataset(X_all, y_all)
n_train, n_val, n_test = 560, 120, 120
train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,
                           drop_last=True,    # drop last incomplete batch (for BatchNorm)
                           num_workers=0)     # num_workers > 0 for parallel loading
val_loader   = DataLoader(val_set,   batch_size=64)
test_loader  = DataLoader(test_set,  batch_size=64)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

# Inspect a batch
X_batch, y_batch = next(iter(train_loader))
print(f"Batch X: {X_batch.shape}  y: {y_batch.shape}")

In [ ]:
# ── 8.2 Complete training loop with early stopping ────────────────────────────
from sklearn.metrics import roc_auc_score

def train_one_epoch(model, loader, optimiser, criterion):
    """One epoch of training. Returns average loss."""
    model.train()                     # ← CRITICAL: enables dropout and batchnorm training mode
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # ── The 5-line training core ───────────────────────────────────────────
        optimiser.zero_grad()                        # 1. clear previous gradients
        y_pred = model(X_batch).squeeze(-1)          # 2. forward pass
        loss   = criterion(y_pred, y_batch)          # 3. compute loss
        loss.backward()                              # 4. backpropagation
        torch.nn.utils.clip_grad_norm_(             # 4b. gradient clipping (prevents exploding)
            model.parameters(), max_norm=1.0)
        optimiser.step()                             # 5. update parameters

        total_loss += loss.item() * len(X_batch)

    return total_loss / len(loader.dataset)


@torch.no_grad()  # disable gradient computation for evaluation
def evaluate(model, loader):
    """Evaluate model. Returns loss and AUC."""
    model.eval()                      # ← CRITICAL: disables dropout, uses running stats for BN
    criterion = nn.BCEWithLogitsLoss()
    all_preds, all_labels, total_loss = [], [], 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch).squeeze(-1)
        loss   = criterion(y_pred, y_batch)
        total_loss += loss.item() * len(X_batch)
        probs = torch.sigmoid(y_pred).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(y_batch.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds)
    return total_loss / len(loader.dataset), auc


# Initialise model and training components
model     = SimpleMLP(in_features=n_feat, hidden=128, out_features=1).to(device)
optimiser = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimiser, T_max=100)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0]).to(device))

# Training loop
best_val_auc  = 0
best_state    = None
patience_cnt  = 0
PATIENCE      = 15
history       = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(1, 101):
    train_loss         = train_one_epoch(model, train_loader, optimiser, criterion)
    val_loss, val_auc  = evaluate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    # Early stopping
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 20 == 0 or epoch == 1:
        lr = optimiser.param_groups[0]['lr']
        print(f"Epoch {epoch:3d} | train={train_loss:.4f} | val_loss={val_loss:.4f} "
              f"| val_AUC={val_auc:.4f} | lr={lr:.2e}")

    if patience_cnt >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

# Load best model and evaluate on test set
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
test_loss, test_auc = evaluate(model, test_loader)
print(f"\nTest AUC: {test_auc:.4f}  (best val AUC was {best_val_auc:.4f})")

In [ ]:
# ── 8.3 Plot training history ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], '#1565C0', lw=2.2, label='Train loss')
axes[0].plot(epochs_range, history['val_loss'],   '#E74C3C', lw=2.2, label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training and Validation Loss', fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.35)

axes[1].plot(epochs_range, history['val_auc'], '#27AE60', lw=2.2)
axes[1].axhline(best_val_auc, color='k', linestyle='--', lw=1.5,
                label=f'Best val AUC = {best_val_auc:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Validation AUC', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.35)

plt.suptitle('Training History — MLP on Synthetic QSAR', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 9 — Saving & Loading Models

In [ ]:
# ── 9.1 Saving and loading ────────────────────────────────────────────────────

# METHOD 1: Save only the state dict (RECOMMENDED)
# state_dict = all learned parameter tensors
torch.save(model.state_dict(), 'mlp_model.pt')

# Load: create model instance first, then load weights
loaded_model = SimpleMLP(in_features=n_feat, hidden=128, out_features=1)
loaded_model.load_state_dict(torch.load('mlp_model.pt', map_location='cpu'))
loaded_model.eval()   # set to eval mode
print("Model loaded from file ✓")

# METHOD 2: Full checkpoint (includes optimiser, epoch, loss, etc.)
checkpoint = {
    'epoch':      epoch,
    'model':      model.state_dict(),
    'optimiser':  optimiser.state_dict(),
    'val_auc':    best_val_auc,
    'config':     {'in_features': n_feat, 'hidden': 128}
}
torch.save(checkpoint, 'checkpoint.pt')

# Restore full training state (to continue training)
ckpt   = torch.load('checkpoint.pt', map_location='cpu')
resume_model = SimpleMLP(in_features=n_feat, hidden=128, out_features=1)
resume_model.load_state_dict(ckpt['model'])
resume_opt = AdamW(resume_model.parameters())
resume_opt.load_state_dict(ckpt['optimiser'])
print(f"Checkpoint loaded | epoch={ckpt['epoch']} | val_auc={ckpt['val_auc']:.4f} ✓")

# METHOD 3: Export to TorchScript (for deployment without Python)
model.eval()
example_input = torch.randn(1, n_feat)
scripted = torch.jit.trace(model.cpu(), example_input)
scripted.save('model_scripted.pt')
print("TorchScript model saved ✓")

---
## Section 10 — GPU Workflow

In [ ]:
# ── 10.1 Moving to GPU ────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# The golden rule: model and data must be on the SAME device
model   = SimpleMLP(n_feat, 128, 1).to(device)       # move model to GPU
X_batch = torch.randn(32, n_feat).to(device)          # move data to GPU

with torch.no_grad():
    output = model(X_batch)                            # runs on GPU

# Move result back to CPU for numpy/sklearn
result_np = output.detach().cpu().numpy()
print(f"Output shape: {output.shape}  on {output.device}")
print(f"CPU numpy: {result_np.shape}")

if torch.cuda.is_available():
    print(f"\nGPU memory allocated: {torch.cuda.memory_allocated()/1e6:.1f} MB")
    print(f"GPU memory cached:    {torch.cuda.memory_reserved()/1e6:.1f} MB")

# Common GPU gotchas:
print("\nCommon GPU errors:")
print("  RuntimeError: Expected all tensors to be on the same device")
print("  → Fix: X.to(device) and model.to(device)")
print()
print("  CUDA out of memory")
print("  → Fix: reduce batch size, use gradient checkpointing, or use AMP (float16)")

In [ ]:
# ── 10.2 Automatic Mixed Precision (AMP) — 2× faster on modern GPUs ──────────
# float16 for forward/loss → float32 for gradients → 2× speedup, 2× memory savings

from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()  # scales loss to prevent float16 underflow

# AMP training loop pattern:
def train_with_amp(model, loader, optimiser, criterion):
    model.train()
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimiser.zero_grad()

        with autocast():                                   # mixed precision context
            y_pred = model(X_batch).squeeze(-1)
            loss   = criterion(y_pred, y_batch)

        scaler.scale(loss).backward()                      # scale loss for fp16
        scaler.unscale_(optimiser)                         # unscale before clip
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimiser)                             # update params
        scaler.update()                                    # adjust scale factor

print("AMP template ready (use this for large models on GPU)")

---
## Section 11 — Common Errors & Debugging

In [ ]:
errors = """
╔══════════════════════════════════════════════════════════════════════════╗
║              PyTorch Common Errors & Fixes                              ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 1: RuntimeError: Sizes of tensors must match                      ║
║   Cause:  Shape mismatch between prediction and target                  ║
║   model(x) → [32, 1] but y → [32]                                      ║
║   Fix:    y_pred = model(x).squeeze(-1)  → [32]                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 2: Loss is NaN from the first step                                ║
║   Cause:  Learning rate too large / exploding gradients                 ║
║   Fix:    Reduce lr, add gradient clipping, check for NaN in input     ║
║           torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)         ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 3: Validation AUC exactly matches training AUC                    ║
║   Cause:  Forgot model.eval() — dropout still active at inference!     ║
║   Fix:    Always call model.eval() before evaluate()                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 4: GPU runs out of memory after the first epoch                   ║
║   Cause:  Accumulating computation graph across batches                 ║
║   Fix:    loss.item() (not loss) for logging                            ║
║           Use torch.no_grad() in validation                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 5: Model does not learn (loss stays constant)                     ║
║   Cause:  Forgot optimiser.zero_grad() — gradients accumulate          ║
║   Cause:  Loss not calling .backward()                                  ║
║   Fix:    zero_grad() → backward() → step() — always in this order     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 6: RuntimeError: Expected all tensors on the same device          ║
║   Cause:  Model on GPU, data on CPU (or vice versa)                    ║
║   Fix:    X = X.to(device) and model = model.to(device)                ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 7: RuntimeError: leaf variable ... moved into the graph           ║
║   Cause:  In-place operation on a tensor that requires grad             ║
║   Fix:    Replace in-place ops: W += lr*grad → W = W + lr*grad         ║
║           Or use torch.no_grad() context for parameter updates          ║
╠══════════════════════════════════════════════════════════════════════════╣
║ CHECKLIST (run through this when something is wrong)                    ║
║  □ model.train() before training loop                                   ║
║  □ model.eval() before validation/test                                  ║
║  □ optimiser.zero_grad() at start of each batch                        ║
║  □ All tensors on same device                                           ║
║  □ loss.backward() before optimiser.step()                             ║
║  □ torch.no_grad() in evaluation                                        ║
║  □ .squeeze(-1) to align pred/target shapes                            ║
║  □ Use loss.item() for logging (not loss itself)                       ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(errors)

---

## What to learn next

| Topic | Resource |
|---|---|
| GNN for molecules | `gnn_toxicology_tutorial.ipynb` in this repo |
| PyTorch Lightning | lightning.ai — cleaner training loops |
| HuggingFace | huggingface.co — pre-trained models |
| PyTorch Geometric | pytorch-geometric.readthedocs.io |
| Official tutorials | pytorch.org/tutorials |
| Deep Learning book | d2l.ai (free, PyTorch edition) |

*Built by Himanshu Goel · [hgoelgithub.github.io](https://hgoelgithub.github.io)*